# Demo 2: Controlled Pendulum
## Demo 2.4: Hybrid - FMPy & OpenSim Co-Simulation

### Description

The following demo implements a controlled pendulum system described in Demo 2.1. The Modelica models for the `Reference`, `AngleEncoder`, `Controller`, and `Drive` are exported as an Co-Simulation FMU using OpenModelica. The FMUs are then imported and simulated in Python using the FMPy package.

The pendulum model for this demo is implemented as an OpenSim model. The OpenSim model is loaded and simulated using the OpenSim API in Python.

### Procedure

**1. Changing the working dirctory to use `SysSimX` package**

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))

from SysSimX.utilities.update_fmus import get_fmu_paths, get_models_within_package, move_file

**2. Get the FMU files**

In [2]:
demo_dir_path = Path(repo_root / "demos" / "ControlledPendulum")
package_path = Path(demo_dir_path / "ControlledPendulum")
package_file_path = Path(demo_dir_path / "ControlledPendulum" / "package.mo")


model_names = get_models_within_package(package_path)
drive = model_names[2]
composed_name = f"ControlledPendulum.{drive}"

IndexError: list index out of range

In [3]:
from OMPython import ModelicaSystem

In [4]:
drive = ModelicaSystem(
    str(package_file_path), composed_name, commandLineOptions="--fmiFlags=s:cvode"
)


Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.
[C:/OM125/OM64bit/OMCompiler/Compiler/BackEnd/Differentiate.mo:268:5-268:157:writable] Error: Derivative of expression "wall_contact = noEvent(pendulum.q_state > wall.q_wall)" w.r.t. "dummyVarC" is non-existent.
Error: Internal error SymbolicJacobian.deriveAll failed
[C:/OM125/OM64bit/OMCompiler/Compiler/BackEnd/SymbolicJacobian.mo:2474:7-2474:74:writable] Error: Internal error SymbolicJacobian.generateSymbolicJacobian failed
[C:/OM125/OM64bit/OMCompiler/Compiler/BackEnd/SymbolicJacobian.mo:2265:9-2265:79:writable] Error: Internal error function createJacobian failed



In [5]:
drive.buildModel()
fmu_path = drive.convertMo2Fmu(fmuType="cs")

[C:/OM125/OM64bit/OMCompiler/Compiler/BackEnd/Differentiate.mo:268:5-268:157:writable] Error: Derivative of expression "wall_contact = noEvent(pendulum.q_state > wall.q_wall)" w.r.t. "dummyVarC" is non-existent.
Error: Internal error SymbolicJacobian.deriveAll failed
[C:/OM125/OM64bit/OMCompiler/Compiler/BackEnd/SymbolicJacobian.mo:2474:7-2474:74:writable] Error: Internal error SymbolicJacobian.generateSymbolicJacobian failed
[C:/OM125/OM64bit/OMCompiler/Compiler/BackEnd/SymbolicJacobian.mo:2265:9-2265:79:writable] Error: Internal error function createJacobian failed



In [6]:
dest_path = Path(demo_dir_path / "FMUs" / "Drive2.fmu")
move_file(fmu_path, dest_path)

In [7]:
fmu_output_dir = Path(demo_dir_path / "FMUs")

fmu_paths = get_fmu_paths(package_path, fmu_output_dir, force_rebuild=False)
for fmu, path in fmu_paths.items():
    print(f"{fmu:<25}: {path}")

All FMUs for package 'ControlledPendulum' already exist. Skipping generation.
AngleEncoder             : c:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\FMUs\AngleEncoder.fmu
Demo_Driven              : c:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\FMUs\Demo_Driven.fmu
Demo_DrivenWithWall      : c:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\FMUs\Demo_DrivenWithWall.fmu
Demo_DrivenWithWallDiscrete: c:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\FMUs\Demo_DrivenWithWallDiscrete.fmu
Demo_UndrivenWallDiscrete: c:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\FMUs\Demo_UndrivenWallDiscrete.fmu
Demo_UndrivenWithWall    : c:\Users\flori\source\repos\FlorianFrech\SystemSimulation\demos\ControlledPendulum\FMUs\Demo_UndrivenWithWall.fmu
Drive                    : c:\Users\flori\source\repos\FlorianFrech\SystemSimulation\

**3. Load the Model Descriptions and Setup Variables**

In [8]:
# Import FMPy
from fmpy import read_model_description

fmu_dict = {}

# Read model descriptions and variable references
for model_name, fmu_path in fmu_paths.items():
    print(30 * "-")
    print(f"Reading FMU: {model_name}")
    fmu_dict[model_name] = {}
    fmu_dict[model_name]["ModelDescription"] = read_model_description(fmu_path)
    vrs = {}
    print("  Variables:")
    for variable in fmu_dict[model_name]["ModelDescription"].modelVariables:
        vrs[variable.name] = variable.valueReference
        print(f"    {variable.name:<30} : {variable.valueReference}")

    fmu_dict[model_name]["VariableReferences"] = vrs

------------------------------
Reading FMU: AngleEncoder
  Variables:
    der(_D_outputAlias_U_q)        : 0
    _D_TMP__D_outputAlias_U_q_4    : 3
    _D_TMP_U_a_4                   : 4
    _D_TMP_alpha_4                 : 5
    U_q                            : 9
    q                              : 11
    nBits                          : 14
    q_max                          : 15
    q_min                          : 16
------------------------------
Reading FMU: Demo_Driven
  Variables:
    drive.I                        : 0
    pendulum.omega_state           : 1
    pendulum.q_state               : 2
    pid.D.x                        : 3
    pid.I.y                        : 4
    der(drive.I)                   : 5
    der(pendulum.omega_state)      : 6
    der(pendulum.q_state)          : 7
    der(pid.D.x)                   : 8
    der(pid.I.y)                   : 9
    drive.torque                   : 14
    pid.u                          : 21
    reference.q_ref                :

In [9]:
# Dictionary to hold variable references
vars_rf = {}
vars_rf["q_ref"] = fmu_dict["Reference"]["VariableReferences"]["q_ref"]

vars_rf["q_state"] = fmu_dict["Pendulum"]["VariableReferences"]["q_state"]
vars_rf["omega_state"] = fmu_dict["Pendulum"]["VariableReferences"]["omega_state"]
vars_rf["torque"] = fmu_dict["Pendulum"]["VariableReferences"]["torque"]

vars_rf["q_sensor"] = fmu_dict["AngleEncoder"]["VariableReferences"]["q"]
vars_rf["U_q"] = fmu_dict["AngleEncoder"]["VariableReferences"]["U_q"]

vars_rf["pid_y"] = fmu_dict["PID_Continuous"]["VariableReferences"]["y"]
vars_rf["pid_ref"] = fmu_dict["PID_Continuous"]["VariableReferences"]["ref"]
vars_rf["pid_u"] = fmu_dict["PID_Continuous"]["VariableReferences"]["u"]

vars_rf["drive_u"] = fmu_dict["Drive"]["VariableReferences"]["u_control"]
vars_rf["drive_omega"] = fmu_dict["Drive"]["VariableReferences"]["omega"]
vars_rf["drive_torque"] = fmu_dict["Drive"]["VariableReferences"]["torque"]

**4. Instantiate the FMUs using `FMU2Slave`**

In [10]:
from fmpy import extract
from fmpy.fmi2 import FMU2Slave

# Instantiate Reference Trajectory
unzip_dir = extract(fmu_paths["Reference"])
ref_fmu = FMU2Slave(
    guid=fmu_dict["Reference"]["ModelDescription"].guid,
    unzipDirectory=unzip_dir,
    modelIdentifier=fmu_dict["Reference"]["ModelDescription"].coSimulation.modelIdentifier,
    instanceName="ref_fmu",
)
ref_fmu.instantiate()

# Instantiate Sensors for Reference Trajectory and Pendulum
unzip_dir = extract(fmu_paths["AngleEncoder"])
sensor_ref_fmu = FMU2Slave(
    guid=fmu_dict["AngleEncoder"]["ModelDescription"].guid,
    unzipDirectory=unzip_dir,
    modelIdentifier=fmu_dict["AngleEncoder"]["ModelDescription"].coSimulation.modelIdentifier,
    instanceName="sensor_ref_fmu",
)
sensor_ref_fmu.instantiate()

sensor_state_fmu = FMU2Slave(
    guid=fmu_dict["AngleEncoder"]["ModelDescription"].guid,
    unzipDirectory=unzip_dir,
    modelIdentifier=fmu_dict["AngleEncoder"]["ModelDescription"].coSimulation.modelIdentifier,
    instanceName="sensor_state_fmu",
)
sensor_state_fmu.instantiate()

# Instantiate Controller
unzip_dir = extract(fmu_paths["PID_Continuous"])
pid_fmu = FMU2Slave(
    guid=fmu_dict["PID_Continuous"]["ModelDescription"].guid,
    unzipDirectory=unzip_dir,
    modelIdentifier=fmu_dict["PID_Continuous"]["ModelDescription"].coSimulation.modelIdentifier,
    instanceName="pid_fmu",
)
pid_fmu.instantiate()

# Instantiate Drive
unzip_dir = extract(fmu_paths["Drive"])
drive = FMU2Slave(
    guid=fmu_dict["Drive"]["ModelDescription"].guid,
    unzipDirectory=unzip_dir,
    modelIdentifier=fmu_dict["Drive"]["ModelDescription"].coSimulation.modelIdentifier,
    instanceName="drive",
)
drive.instantiate()

fmu_list = [ref_fmu, sensor_ref_fmu, sensor_state_fmu, pid_fmu, drive]

**5. Load the OpenSim Pendulum Model**

In [11]:
from OpenSim.opensim_pendulum import PendulumModel

pendulum_model = PendulumModel()

**6. FMU Setup and Initialization**

In [12]:
t = 0.0
tf = 10.0
h = 0.001

for fmu in fmu_list:
    fmu.setupExperiment(startTime=t)
    fmu.enterInitializationMode()
    fmu.exitInitializationMode()

# Initialize logging arrays
ts = []
q_ref_log, q_state_log, omega_state_log = [], [], []
U_ref_log, U_state_log = [], []

**7. Simulation Loop**

In [13]:
while t < tf:
    # 1) Read the current reference position
    ref_fmu.doStep(t, h)
    q_ref = ref_fmu.getReal([vars_rf["q_ref"]])

    # 2) Read the plant state at the current time
    q_state = [pendulum_model.get_q()]
    omega_state = [pendulum_model.get_omega()]

    # 3) Sensors: set inputs -> step -> read outputs
    sensor_ref_fmu.setReal([vars_rf["q_sensor"]], q_ref)
    sensor_state_fmu.setReal([vars_rf["q_sensor"]], q_state)
    sensor_ref_fmu.doStep(t, h)
    sensor_state_fmu.doStep(t, h)
    U_q_ref = sensor_ref_fmu.getReal([vars_rf["U_q"]])
    U_q_state = sensor_state_fmu.getReal([vars_rf["U_q"]])

    # 4) Controller: set inputs -> step -> read outputs
    pid_fmu.setReal([vars_rf["pid_y"], vars_rf["pid_ref"]], [U_q_state[0], U_q_ref[0]])
    pid_fmu.doStep(currentCommunicationPoint=t, communicationStepSize=h)
    u_pid = pid_fmu.getReal([vars_rf["pid_u"]])

    # 5) Drive: set inputs -> step -> read outputs
    drive.setReal([vars_rf["drive_u"], vars_rf["drive_omega"]], [u_pid[0], omega_state[0]])
    drive.doStep(currentCommunicationPoint=t, communicationStepSize=h)
    torque = drive.getReal([vars_rf["drive_torque"]])[0]

    # 6) Plant: set inputs -> step
    pendulum_model.set_torque(torque)
    pendulum_model.do_step(t, h)

    # Log data
    ts.append(t)
    q_ref_log.append(q_ref[0])
    q_state_log.append(q_state[0])
    omega_state_log.append(omega_state[0])
    U_ref_log.append(U_q_ref[0])
    U_state_log.append(U_q_state[0])

    # Advance time
    t += h

**8. Plotting the Results**

In [14]:
# Create a simple plotly figure for the q_ref and q_state over time
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ts,
        y=q_ref_log,
        mode="lines",
        name="Reference Angle (q_ref)",
        line=dict(color="white", dash="dash"),
    )
)
fig.add_trace(
    go.Scatter(
        x=ts, y=q_state_log, mode="lines", name="State Angle (q_state)", line=dict(color="red")
    )
)
fig.update_layout(
    title="Pendulum Angle Tracking",
    xaxis_title="Time (s)",
    yaxis_title="Angle (degrees)",
    legend_title="Legend",
    template="plotly_dark",
)
fig.show()

In [15]:
from opensim import RowVector, STOFileAdapter, TimeSeriesTable

table = TimeSeriesTable()
table.setColumnLabels(["q", "q_dot"])

for i in range(len(ts)):
    row = RowVector(2)
    row[0] = q_state_log[i]
    row[1] = omega_state_log[i]

    # appendRow takes time as first argument, then the row data
    table.appendRow(ts[i], row)

# Use STOFileAdapter to write the table
adapter = STOFileAdapter()
adapter.write(table, r"OpenSim/demo_hybrid_results.sto")